In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/dataset_clean.csv')

print('Размер датасета:', df.shape)

Размер датасета: (113999, 21)


In [3]:
features = [
    'danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature'
]

target = 'track_genre'

In [4]:
track_genre_count = (
    df.groupby('track_id')['track_genre']
      .nunique()
)

single_genre_ids = track_genre_count[
    track_genre_count == 1
].index

df_single = df[
    df['track_id'].isin(single_genre_ids)
].copy()

print('Размер df_single:', df_single.shape)
print('Уникальных track_id:', df_single['track_id'].nunique())
print('Количество жанров:', df_single['track_genre'].nunique())

Размер df_single: (73789, 21)
Уникальных track_id: 73441
Количество жанров: 112


In [5]:
X = df_single[features]
y = df_single[target]

print('X:', X.shape)
print('y:', y.shape)

X: (73789, 12)
y: (73789,)


In [6]:
from sklearn.model_selection import train_test_split

# 20% оставляем для финального тестирования
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Из оставшихся 80% выделяем 20% на validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

print('Train:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Train: (47224, 12)
Validation: (11807, 12)
Test: (14758, 12)


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logreg.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](112,)","['acoustic','afrobeat','alt-rock',...,'trip-hop','turkish','world-music']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['danceability','energy','key',...,'valence','tempo','time_signature']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [12]:
from sklearn.metrics import classification_report

report = classification_report(
    y_val,
    y_val_pred,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

report_df.sort_values('f1-score').head(15)

,precision,recall,f1-score,support
alternative,0.0,0.0,0.0,25.0
alt-rock,0.0,0.0,0.0,25.0
anime,0.0,0.0,0.0,106.0
blues,0.0,0.0,0.0,62.0
brazil,0.0,0.0,0.0,45.0
british,0.0,0.0,0.0,92.0
edm,0.0,0.0,0.0,21.0
electro,0.0,0.0,0.0,55.0
house,0.0,0.0,0.0,30.0
j-pop,0.0,0.0,0.0,56.0


In [13]:
report_df.sort_values('f1-score', ascending=False).head(15)

,precision,recall,f1-score,support
comedy,0.750000,0.867925,0.804665,159.0
sleep,0.594059,0.750000,0.662983,160.0
grindcore,0.427386,0.651899,0.516291,158.0
study,0.402256,0.668750,0.502347,160.0
pagode,0.347107,0.403846,0.373333,104.0
detroit-techno,0.321429,0.414474,0.362069,152.0
tango,0.303030,0.437500,0.358056,160.0
romance,0.275986,0.481250,0.350797,160.0
honky-tonk,0.237410,0.618750,0.343154,160.0
opera,0.262295,0.474074,0.337731,135.0


Теперь следующий логичный эксперимент — Random Forest.

Почему именно он:

Logistic Regression строит линейные границы;
Random Forest способен моделировать нелинейные зависимости между energy, danceability, loudness, acousticness и т. д.;
он не требует StandardScaler;
мы сможем честно сравнить две разные модели на одних и тех же train/validation данных.

Гипотеза:
Проблема Logistic Regression в том, что зависимости между аудиопризнаками и жанром слишком нелинейны?

Если Random Forest заметно поднимет Macro-F1 — это будет аргумент в пользу этой гипотезы.

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [15]:
y_val_pred_rf = rf.predict(X_val)

accuracy_rf = accuracy_score(y_val, y_val_pred_rf)
f1_macro_rf = f1_score(y_val, y_val_pred_rf, average='macro')
f1_weighted_rf = f1_score(y_val, y_val_pred_rf, average='weighted')

print(f'Accuracy:     {accuracy_rf:.4f}')
print(f'Macro-F1:     {f1_macro_rf:.4f}')
print(f'Weighted-F1:  {f1_weighted_rf:.4f}')

Accuracy:     0.3288
Macro-F1:     0.2875
Weighted-F1:  0.3108


In [16]:
results = pd.DataFrame({
    'Model': [
        'Dummy',
        'Logistic Regression',
        'Random Forest'
    ],
    'Accuracy': [
        0.0136,
        0.2104,
        accuracy_rf
    ],
    'Macro-F1': [
        0.0002,
        0.1361,
        f1_macro_rf
    ],
    'Weighted-F1': [
        0.0004,
        0.1759,
        f1_weighted_rf
    ]
})

results

,Model,Accuracy,Macro-F1,Weighted-F1
0,Dummy,0.013600,0.00020,0.000400
1,Logistic Regression,0.210400,0.13610,0.175900
2,Random Forest,0.328788,0.28746,0.310761


XGBoost — следующий основной эксперимент

Для такого табличного multiclass-классификатора это очень естественный следующий кандидат.

In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train)
y_val_xgb = label_encoder.transform(y_val)

print('Количество классов:', len(label_encoder.classes_))
print('Пример:')
for i in range(10):
    print(i, '->', label_encoder.classes_[i])

Количество классов: 112
Пример:
0 -> acoustic
1 -> afrobeat
2 -> alt-rock
3 -> alternative
4 -> ambient
5 -> anime
6 -> black-metal
7 -> bluegrass
8 -> blues
9 -> brazil


In [24]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(label_encoder.classes_),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

In [25]:
xgb.fit(X_train, y_train_xgb)


,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) r

In [26]:
y_val_pred_xgb_encoded = xgb.predict(X_val)

y_val_pred_xgb = label_encoder.inverse_transform(
    y_val_pred_xgb_encoded.astype(int)
)

In [27]:
accuracy_xgb = accuracy_score(y_val, y_val_pred_xgb)
f1_macro_xgb = f1_score(
    y_val,
    y_val_pred_xgb,
    average='macro'
)
f1_weighted_xgb = f1_score(
    y_val,
    y_val_pred_xgb,
    average='weighted'
)

print(f'Accuracy:     {accuracy_xgb:.4f}')
print(f'Macro-F1:     {f1_macro_xgb:.4f}')
print(f'Weighted-F1:  {f1_weighted_xgb:.4f}')

Accuracy:     0.3265
Macro-F1:     0.2941
Weighted-F1:  0.3173


In [35]:
final_encoder = LabelEncoder()

y_train_final_encoded = final_encoder.fit_transform(y_train_final)
y_test_encoded = final_encoder.transform(y_test)

print(len(final_encoder.classes_))

112


In [52]:
final_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(final_encoder.classes_),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

обучаем на трейне

In [37]:
final_model.fit(
    X_train_final,
    y_train_final_encoded
)

,"objective objective: str | xgboost.objective.Objective | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) r

In [38]:
y_test_pred_encoded = final_model.predict(X_test)

y_test_pred = final_encoder.inverse_transform(
    y_test_pred_encoded.astype(int)
)

In [39]:
test_accuracy = accuracy_score(y_test, y_test_pred)

test_f1_macro = f1_score(
    y_test,
    y_test_pred,
    average='macro'
)

test_f1_weighted = f1_score(
    y_test,
    y_test_pred,
    average='weighted'
)

print(f'Test Accuracy:    {test_accuracy:.4f}')
print(f'Test Macro-F1:    {test_f1_macro:.4f}')
print(f'Test Weighted-F1: {test_f1_weighted:.4f}')

Test Accuracy:    0.3415
Test Macro-F1:    0.3132
Test Weighted-F1: 0.3332


изучаем ошибки и взаимосвязи

In [45]:
error_analysis = X_test.copy()

error_analysis["true_genre"] = y_test.values
error_analysis["predicted_genre"] = y_test_pred

error_analysis["correct"] = (
    error_analysis["true_genre"] == error_analysis["predicted_genre"]
)

error_analysis.head()

,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,true_genre,predicted_genre,correct
19859,0.529,0.496,7,-9.007,1,0.0290,0.173000,0.00000,0.2510,0.278,136.859,4,country,cantopop,False
21179,0.515,0.612,7,-6.314,1,0.0600,0.034400,0.00000,0.5900,0.401,126.106,5,dancehall,progressive-house,False
48017,0.900,0.697,9,-3.439,1,0.2820,0.146000,0.00000,0.2870,0.670,93.940,4,hardcore,hardcore,True
73988,0.899,0.868,7,-7.334,1,0.0674,0.000525,0.00185,0.0597,0.735,126.992,4,minimal-techno,chicago-house,False
22675,0.326,0.979,5,-8.173,1,0.1530,0.005640,0.32100,0.2630,0.429,90.768,4,death-metal,grindcore,False


In [46]:
genre_accuracy = (
    error_analysis
    .groupby("true_genre")["correct"]
    .mean()
    .sort_values()
)

genre_accuracy.head(15)

true_genre
alt-rock      0.000000
edm           0.000000
punk-rock     0.000000
metal         0.000000
reggaeton     0.000000
brazil        0.017857
latino        0.034483
j-rock        0.043478
mpb           0.044444
electronic    0.050360
emo           0.063636
psych-rock    0.064815
blues         0.064935
groove        0.065421
punk          0.071429
Name: correct, dtype: float64

In [47]:
genre_accuracy.tail(15).sort_values(ascending=False)

true_genre
sleep            0.800000
study            0.760000
drum-and-bass    0.747312
comedy           0.738693
jazz             0.714286
grindcore        0.681818
hardstyle        0.666667
alternative      0.656250
dance            0.637363
tango            0.635000
honky-tonk       0.570000
electro          0.565217
romance          0.560000
salsa            0.550505
party            0.545455
Name: correct, dtype: float64

In [48]:
confusions = (
    error_analysis[
        error_analysis["true_genre"] != error_analysis["predicted_genre"]
    ]
    .groupby(["true_genre", "predicted_genre"])
    .size()
    .sort_values(ascending=False)
)

confusions.head(20)

true_genre      predicted_genre
chicago-house   detroit-techno     35
detroit-techno  chicago-house      33
romance         tango              27
minimal-techno  detroit-techno     26
disney          show-tunes         26
grindcore       black-metal        25
samba           pagode             25
black-metal     grindcore          25
gospel          world-music        24
death-metal     grindcore          24
mandopop        cantopop           23
power-pop       garage             23
idm             iranian            22
ambient         new-age            22
cantopop        mandopop           22
death-metal     heavy-metal        22
dancehall       j-dance            21
acoustic        cantopop           20
gospel          mandopop           20
idm             detroit-techno     20
dtype: int64

In [49]:
confusion_rates = (
    error_analysis[
        error_analysis["true_genre"] != error_analysis["predicted_genre"]
    ]
    .groupby(["true_genre", "predicted_genre"])
    .size()
    .reset_index(name="errors")
)

genre_counts = (
    error_analysis
    .groupby("true_genre")
    .size()
    .reset_index(name="total")
)

confusion_rates = confusion_rates.merge(
    genre_counts,
    on="true_genre"
)

confusion_rates["error_rate"] = (
    confusion_rates["errors"] / confusion_rates["total"]
)

confusion_rates = confusion_rates.sort_values(
    "error_rate",
    ascending=False
)

confusion_rates.head(20)

,true_genre,predicted_genre,errors,total,error_rate
3617,techno,minimal-techno,20,78,0.256410
2363,latino,dancehall,7,29,0.241379
3005,reggaeton,kids,3,14,0.214286
2947,punk-rock,power-pop,10,49,0.204082
2478,metal,grunge,6,30,0.200000
2514,minimal-techno,detroit-techno,26,130,0.200000
3222,samba,pagode,25,126,0.198413
2917,punk,power-pop,8,42,0.190476
1460,gospel,world-music,24,127,0.188976
493,chicago-house,detroit-techno,35,190,0.184211


In [50]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_test_pred,
    output_dict=True,
    zero_division=0
)

report_df = (
    pd.DataFrame(report)
    .T
    .reset_index()
    .rename(columns={"index": "genre"})
)

report_df = report_df[
    ~report_df["genre"].isin(["accuracy", "macro avg", "weighted avg"])
]

report_df = report_df.sort_values("f1-score", ascending=False)

report_df.head(15)

,genre,precision,recall,f1-score,support
18,comedy,0.924528,0.738693,0.821229,199.0
100,sleep,0.808081,0.800000,0.804020,200.0
64,jazz,0.902778,0.714286,0.797546,91.0
3,alternative,1.000000,0.656250,0.792453,32.0
27,drum-and-bass,0.634703,0.747312,0.686420,186.0
31,electro,0.829787,0.565217,0.672414,69.0
20,dance,0.707317,0.637363,0.670520,91.0
90,rock,0.903226,0.509091,0.651163,55.0
53,house,0.833333,0.526316,0.645161,38.0
103,study,0.556777,0.760000,0.642706,200.0


In [51]:
report_df.tail(15)

,genre,precision,recall,f1-score,support
110,turkish,0.089362,0.120690,0.102689,174.0
40,gospel,0.108108,0.094488,0.100840,127.0
43,groove,0.218750,0.065421,0.100719,107.0
84,psych-rock,0.145833,0.064815,0.089744,108.0
33,emo,0.104478,0.063636,0.079096,110.0
32,electronic,0.114754,0.050360,0.070000,139.0
74,mpb,0.083333,0.044444,0.057971,90.0
63,j-rock,0.085714,0.043478,0.057692,69.0
68,latino,0.125000,0.034483,0.054054,29.0
9,brazil,0.076923,0.017857,0.028986,56.0


Что мы уже выяснили
1. Модель хорошо распознаёт отдельные достаточно характерные классы
2. А вот близкие жанры действительно смешиваются
chicago-house → detroit-techno : 35
detroit-techno → chicago-house : 33
Например, модель умеет улавливать определённую структуру металлических жанров, но не может надёжно провести границу между некоторыми поджанрами.


importance XGBoost

In [60]:
final_model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(final_encoder.classes_),
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_train_final,
    y_train_final_encoded
)

print("Модель обучена")

Модель обучена


In [61]:
xgb_importance = pd.Series(
    final_model.feature_importances_,
    index=features
).sort_values(ascending=False)

xgb_importance

acousticness        0.108061
instrumentalness    0.107118
danceability        0.093756
mode                0.090104
energy              0.084531
valence             0.083113
loudness            0.081465
tempo               0.079969
speechiness         0.079620
time_signature      0.075936
liveness            0.062572
key                 0.053757
dtype: float32

То есть XGBoost использует практически все 12 признаков.

In [65]:
from sklearn.metrics import accuracy_score, f1_score

y_pred = final_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test_encoded,
    y_pred
)

baseline_macro_f1 = f1_score(
    y_test_encoded,
    y_pred,
    average="macro"
)

baseline_weighted_f1 = f1_score(
    y_test_encoded,
    y_pred,
    average="weighted"
)

print(f"Baseline Accuracy:     {baseline_accuracy:.4f}")
print(f"Baseline Macro-F1:     {baseline_macro_f1:.4f}")
print(f"Baseline Weighted-F1:  {baseline_weighted_f1:.4f}")

Baseline Accuracy:     0.3415
Baseline Macro-F1:     0.3132
Baseline Weighted-F1:  0.3332


In [83]:
def train_and_evaluate(features_subset):
    # Выбираем нужные признаки
    X_train_subset = X_train_final[features_subset]
    X_test_subset = X_test[features_subset]

    # Создаём модель
    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.08,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=len(final_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    # Обучение
    model.fit(
        X_train_subset,
        y_train_final_encoded
    )

    # Предсказание
    y_pred = model.predict(X_test_subset)

    # Метрики
    return {
        "Accuracy": accuracy_score(
            y_test_encoded,
            y_pred
        ),
        "Macro-F1": f1_score(
            y_test_encoded,
            y_pred,
            average="macro"
        ),
        "Weighted-F1": f1_score(
            y_test_encoded,
            y_pred,
            average="weighted"
        )
    }

In [84]:
results = {}

# 1. Все признаки
results["All features"] = train_and_evaluate(features)

# 2. Без loudness
features_no_loudness = [
    f for f in features
    if f != "loudness"
]
results["Without loudness"] = train_and_evaluate(
    features_no_loudness
)

# 3. Без energy
features_no_energy = [
    f for f in features
    if f != "energy"
]
results["Without energy"] = train_and_evaluate(
    features_no_energy
)

# 4. Без energy + loudness
features_no_energy_loudness = [
    f for f in features
    if f not in ["energy", "loudness"]
]
results["Without energy + loudness"] = train_and_evaluate(
    features_no_energy_loudness
)

# Таблица результатов
results_df = pd.DataFrame(results).T
results_df

,Accuracy,Macro-F1,Weighted-F1
All features,0.341510,0.313156,0.333160
Without loudness,0.325789,0.300305,0.318039
Without energy,0.325789,0.300788,0.317927
Without energy + loudness,0.302819,0.278849,0.294325


In [82]:
print("X_train:", X_train.shape)
print("X_train_final:", X_train_final.shape)
print("y_train_final_encoded:", y_train_final_encoded.shape)

print("\nX_test:", X_test.shape)
print("y_test_encoded:", y_test_encoded.shape)

print("\nfinal_model.n_features_in_:", final_model.n_features_in_)

X_train: (47224, 12)
X_train_final: (59031, 12)
y_train_final_encoded: (59031,)

X_test: (14758, 12)
y_test_encoded: (14758,)

final_model.n_features_in_: 12


То есть, несмотря на высокую корреляцию energy и loudness, удалять их из финального набора признаков не стоит.

In [85]:
y_pred = final_model.predict(X_test)

print("Количество предсказаний:", len(y_pred))
print("Количество истинных ответов:", len(y_test_encoded))

Количество предсказаний: 14758
Количество истинных ответов: 14758


In [86]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test_encoded,
    y_pred,
    target_names=final_encoder.classes_,
    output_dict=True
)

report_df = pd.DataFrame(report).T

report_df.head()

,precision,recall,f1-score,support
acoustic,0.184358,0.201220,0.192420,164.0
afrobeat,0.200980,0.216931,0.208651,189.0
alt-rock,0.000000,0.000000,0.000000,32.0
alternative,1.000000,0.656250,0.792453,32.0
ambient,0.279070,0.251748,0.264706,143.0


ТОП-10:


,precision,recall,f1-score,support
comedy,0.924528,0.738693,0.821229,199.0
sleep,0.808081,0.800000,0.804020,200.0
jazz,0.902778,0.714286,0.797546,91.0
alternative,1.000000,0.656250,0.792453,32.0
drum-and-bass,0.634703,0.747312,0.686420,186.0
electro,0.829787,0.565217,0.672414,69.0
dance,0.707317,0.637363,0.670520,91.0
rock,0.903226,0.509091,0.651163,55.0
house,0.833333,0.526316,0.645161,38.0
study,0.556777,0.760000,0.642706,200.0



ХУДШИЕ-10:


,precision,recall,f1-score,support
electronic,0.114754,0.050360,0.070000,139.0
mpb,0.083333,0.044444,0.057971,90.0
j-rock,0.085714,0.043478,0.057692,69.0
latino,0.125000,0.034483,0.054054,29.0
brazil,0.076923,0.017857,0.028986,56.0
alt-rock,0.000000,0.000000,0.000000,32.0
edm,0.000000,0.000000,0.000000,26.0
metal,0.000000,0.000000,0.000000,30.0
reggaeton,0.000000,0.000000,0.000000,14.0
punk-rock,0.000000,0.000000,0.000000,49.0



САМЫЕ МАЛЕНЬКИЕ КЛАССЫ:


,precision,recall,f1-score,support
reggaeton,0.000000,0.000000,0.000000,14.0
reggae,0.833333,0.294118,0.434783,17.0
indie,0.900000,0.375000,0.529412,24.0
edm,0.000000,0.000000,0.000000,26.0
latino,0.125000,0.034483,0.054054,29.0
metal,0.000000,0.000000,0.000000,30.0
alternative,1.000000,0.656250,0.792453,32.0
alt-rock,0.000000,0.000000,0.000000,32.0
house,0.833333,0.526316,0.645161,38.0
dub,0.162162,0.150000,0.155844,40.0


найдём 10 самых частых ошибок между конкретными жанрами.

,True genre,Predicted genre,Count
493,chicago-house,detroit-techno,35
861,detroit-techno,chicago-house,33
3135,romance,tango,27
2514,minimal-techno,detroit-techno,26
962,disney,show-tunes,26
1513,grindcore,black-metal,25
3222,samba,pagode,25
225,black-metal,grindcore,25
1460,gospel,world-music,24
817,death-metal,grindcore,24


Ошибки модели преимущественно сосредоточены между близкими музыкальными жанрами и поджанрами. Например, наиболее частые ошибки наблюдаются между Chicago House и Detroit Techno, Grindcore и Black Metal, а также Mandopop и Cantopop. Это показывает, что выбранные аудиопризнаки позволяют выделять общие характеристики жанров, но хуже подходят для разделения стилистически близких классов.